In [0]:
from pyspark.sql.functions import col, avg, count, datediff, sum, when, current_date, max

orders = spark.read.table('workspace.ecommerce_gold.orders')
customers = spark.read.table('workspace.ecommerce_gold.customers')
order_items = spark.read.table('workspace.ecommerce_silver.order_items')
reviews = spark.read.table('workspace.ecommerce_silver.order_reviews')


order_features = orders.groupBy('customer_id').agg(
    count('order_id').alias('total_order'),
    avg(datediff(col('order_delivered_customer_date'), col('order_purchase_timestamp'))).alias('avg_delivery_days'),
    avg(when(col('order_delivered_customer_date') > col('order_estimated_delivery_date'), 1).otherwise(0)).alias('late_delivery_rate')
)

order_items_agg = order_items.groupBy('order_id').agg(
    sum('price').alias('order_total_value'),
    count('order_item_id').alias('order_total_items')
)

customer_items_features = orders.select('order_id', 'customer_id') \
    .join(order_items_agg, 'order_id') \
        .groupBy('customer_id').agg(
            avg('order_total_value').alias('avg_order_value'),
            sum('order_total_items').alias('total_items')
        )

review_features = orders.select('order_id', 'customer_id') \
    .join(reviews, 'order_id') \
    .groupBy('customer_id').agg(
        avg('review_score').alias('avg_review_score'),
        count('review_id').alias('review_count')
    )

recency_feature = orders.groupBy("customer_id").agg(
    datediff(
        current_date(),
        max("order_purchase_timestamp")
    ).alias("days_since_last_order")
)

customer_monetary = orders.select('order_id', 'customer_id') \
    .join(order_items_agg, 'order_id') \
    .groupBy('customer_id') \
    .agg(sum('order_total_value').alias('total_spent'))

ml_customer = order_features \
    .join(recency_feature, 'customer_id', 'left') \
    .join(customer_items_features, 'customer_id', 'left') \
    .join(review_features, 'customer_id', 'left') \
    .join(customer_monetary, 'customer_id', 'left') \
    .fillna(0)      

In [0]:
ml_customer = ml_customer.withColumn(
    "high_value",
    when(col("total_spent") > 100, 1).otherwise(0)
)


In [0]:
from pyspark.ml.feature import VectorAssembler

features_cols = [
    'total_order',
    'avg_delivery_days',
    'late_delivery_rate',
    'total_items',
    'avg_review_score',
    'review_count',
    'days_since_last_order'
]

assembler = VectorAssembler(
    inputCols=features_cols,
    outputCol='features'
)

ml_ready = assembler.transform(ml_customer) \
                     .select('features', 'high_value')


In [0]:
ml_customer.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.ecommerce_ml.customer_features")


In [0]:
train_df, test_df = ml_ready.randomSplit([0.8, 0.2], seed = 42)

In [0]:

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

lr = LogisticRegression(
    labelCol= 'high_value',
    featuresCol='features'
)
model = lr.fit(train_df)

predictions = model.transform(test_df)

evaluator = BinaryClassificationEvaluator(
    labelCol='high_value',
    metricName= "areaUnderROC"
)
auc = evaluator.evaluate(predictions)

print(f"AUC: {auc}")